# Arrays & Strings — Senior Interview Patterns

Adobe Interview Prep | 14 YOE

Covers:

  1. Two Pointers

  2. Sliding Window (fixed & variable)

  3. Prefix Sum

  4. Kadane's Algorithm (max subarray)

  5. Boyer-Moore Majority Vote

  6. Dutch National Flag (3-way partition)

  7. Merge Intervals

  8. String: longest substring without repeating chars

  9. String: minimum window substring

  10. Matrix: spiral order, rotate 90°

In [ ]:
from __future__ import annotations
from collections import defaultdict, Counter

## 1. Two Pointers

**Concept:** Maintain two indices moving toward each other or in the same direction to avoid O(n²) nested loops. Classic for sorted-array pair problems.  

**Use when:** Pair sum in sorted array, three-sum, container with most water.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def two_sum_sorted(nums: list[int], target: int) -> tuple[int, int]:
    """
    Find indices of two numbers in a SORTED array that sum to target.
    Time O(n)  Space O(1)

    >>> two_sum_sorted([2, 7, 11, 15], 9)
    (0, 1)
    """
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        s = nums[lo] + nums[hi]
        if s == target:
            return lo, hi
        elif s < target:
            lo += 1
        else:
            hi -= 1
    return -1, -1

# ── demo ──
print(two_sum_sorted([2, 7, 11, 15], 9))

In [ ]:
def three_sum(nums: list[int]) -> list[list[int]]:
    """
    All unique triplets summing to 0.
    Time O(n²)  Space O(1) output aside.

    >>> three_sum([-1, 0, 1, 2, -1, -4])
    [[-1, -1, 2], [-1, 0, 1]]
    """
    nums.sort()
    result = []
    for i, a in enumerate(nums):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            s = a + nums[lo] + nums[hi]
            if s == 0:
                result.append([a, nums[lo], nums[hi]])
                while lo < hi and nums[lo] == nums[lo + 1]: lo += 1
                while lo < hi and nums[hi] == nums[hi - 1]: hi -= 1
                lo += 1; hi -= 1
            elif s < 0:
                lo += 1
            else:
                hi -= 1
    return result

# ── demo ──
print(three_sum([-1, 0, 1, 2, -1, -4]))

In [ ]:
def container_with_most_water(heights: list[int]) -> int:
    """
    Largest rectangle formed by two vertical lines.
    Time O(n)  Space O(1)

    >>> container_with_most_water([1,8,6,2,5,4,8,3,7])
    49
    """
    lo, hi, best = 0, len(heights) - 1, 0
    while lo < hi:
        water = min(heights[lo], heights[hi]) * (hi - lo)
        best = max(best, water)
        if heights[lo] < heights[hi]:
            lo += 1
        else:
            hi -= 1
    return best

# ── demo ──
print(container_with_most_water([1,8,6,2,5,4,8,3,7]))

## 2. Sliding Window

**Concept:** Expand/shrink a window over an array to track a running aggregate without recomputation. Avoids O(n²) brute force.  

**Use when:** Maximum/minimum subarray of fixed size, longest subarray with a constraint, fruit into baskets.  

**Time:** O(n) fixed window · O(n) amortized variable window &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def max_sum_subarray_k(nums: list[int], k: int) -> int:
    """
    Fixed window: max sum of any contiguous subarray of length k.
    Time O(n)  Space O(1)

    >>> max_sum_subarray_k([2,1,5,1,3,2], 3)
    9
    """
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]
        best = max(best, window)
    return best

# ── demo ──
print(max_sum_subarray_k([2,1,5,1,3,2], 3))

In [ ]:
def longest_subarray_at_most_k_zeros(nums: list[int], k: int) -> int:
    """
    Variable window: longest subarray containing at most k zeros.
    LeetCode 1004 — Max Consecutive Ones III.
    Time O(n)  Space O(1)

    >>> longest_subarray_at_most_k_zeros([1,1,1,0,0,0,1,1,1,1,0], 2)
    6
    """
    lo = zeros = best = 0
    for hi in range(len(nums)):
        if nums[hi] == 0:
            zeros += 1
        while zeros > k:
            if nums[lo] == 0:
                zeros -= 1
            lo += 1
        best = max(best, hi - lo + 1)
    return best

# ── demo ──
print(longest_subarray_at_most_k_zeros([1,1,1,0,0,0,1,1,1,1,0], 2))

In [ ]:
def fruits_into_baskets(fruits: list[int]) -> int:
    """
    Variable window: max subarray with at most 2 distinct values.
    Time O(n)  Space O(1)

    >>> fruits_into_baskets([3,3,3,1,2,1,1,2,3,3,4])
    5
    """
    freq: dict[int, int] = {}
    lo = best = 0
    for hi, fruit in enumerate(fruits):
        freq[fruit] = freq.get(fruit, 0) + 1
        while len(freq) > 2:
            freq[fruits[lo]] -= 1
            if freq[fruits[lo]] == 0:
                del freq[fruits[lo]]
            lo += 1
        best = max(best, hi - lo + 1)
    return best

# ── demo ──
print(fruits_into_baskets([3,3,3,1,2,1,1,2,3,3,4]))

## 3. Prefix Sum

**Concept:** Precompute cumulative sums so any subarray sum `[i..j]` = `prefix[j+1] - prefix[i]` in O(1).  

**Use when:** Multiple range-sum queries, counting subarrays with a target sum, product except self.  

**Time:** O(n) build · O(1) query &nbsp;|&nbsp; **Space:** O(n)

In [ ]:
def subarray_sum_equals_k(nums: list[int], k: int) -> int:
    """
    Count subarrays whose elements sum exactly to k.
    Time O(n)  Space O(n)

    >>> subarray_sum_equals_k([1, 1, 1], 2)
    2
    >>> subarray_sum_equals_k([1, 2, 3], 3)
    2
    """
    count = 0
    prefix = 0
    seen = defaultdict(int)
    seen[0] = 1
    for n in nums:
        prefix += n
        count += seen[prefix - k]
        seen[prefix] += 1
    return count

# ── demo ──
print(subarray_sum_equals_k([1, 1, 1], 2))
print(subarray_sum_equals_k([1, 2, 3], 3))

In [ ]:
def product_except_self(nums: list[int]) -> list[int]:
    """
    Output[i] = product of all elements except nums[i], no division.
    Time O(n)  Space O(1) extra (output array excluded).

    >>> product_except_self([1,2,3,4])
    [24, 12, 8, 6]
    """
    n = len(nums)
    out = [1] * n
    prefix = 1
    for i in range(n):
        out[i] = prefix
        prefix *= nums[i]
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out

# ── demo ──
print(product_except_self([1,2,3,4]))

## 4. Kadane's Algorithm

**Concept:** Track the maximum subarray ending at each position — either extend the previous subarray or start fresh at the current element.  

**Use when:** Maximum sum contiguous subarray, any variant needing the best suffix at each position.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def max_subarray(nums: list[int]) -> int:
    """
    Maximum sum contiguous subarray (Kadane).
    Time O(n)  Space O(1)

    >>> max_subarray([-2,1,-3,4,-1,2,1,-5,4])
    6
    """
    best = cur = nums[0]
    for n in nums[1:]:
        cur = max(n, cur + n)
        best = max(best, cur)
    return best

# ── demo ──
print(max_subarray([-2,1,-3,4,-1,2,1,-5,4]))

In [ ]:
def max_subarray_with_indices(nums: list[int]) -> tuple[int, int, int]:
    """
    Returns (max_sum, start_index, end_index).

    >>> max_subarray_with_indices([-2,1,-3,4,-1,2,1,-5,4])
    (6, 3, 6)
    """
    best = cur = nums[0]
    start = end = 0
    temp_start = 0
    for i in range(1, len(nums)):
        if nums[i] > cur + nums[i]:
            cur = nums[i]
            temp_start = i
        else:
            cur += nums[i]
        if cur > best:
            best = cur
            start = temp_start
            end = i
    return best, start, end

# ── demo ──
print(max_subarray_with_indices([-2,1,-3,4,-1,2,1,-5,4]))

## 5. Boyer-Moore Majority Vote

**Concept:** Single-pass cancellation trick — maintain a candidate and a count; when count hits 0, replace the candidate. Extend to n/3 with two candidates.  

**Use when:** Find element appearing > n/2 times, or all elements appearing > n/3 times.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def majority_element(nums: list[int]) -> int:
    """
    Element that appears more than n/2 times (guaranteed to exist).
    Time O(n)  Space O(1)

    >>> majority_element([3,2,3])
    3
    >>> majority_element([2,2,1,1,1,2,2])
    2
    """
    candidate = count = 0
    for n in nums:
        if count == 0:
            candidate = n
        count += 1 if n == candidate else -1
    return candidate

# ── demo ──
print(majority_element([3,2,3]))
print(majority_element([2,2,1,1,1,2,2]))

In [ ]:
def majority_element_n3(nums: list[int]) -> list[int]:
    """
    All elements appearing more than n/3 times (at most 2 such elements).
    Time O(n)  Space O(1)

    >>> sorted(majority_element_n3([3,2,3]))
    [3]
    >>> sorted(majority_element_n3([1,1,1,3,3,2,2,2]))
    [1, 2]
    """
    c1 = c2 = None
    cnt1 = cnt2 = 0
    for n in nums:
        if n == c1:       cnt1 += 1
        elif n == c2:     cnt2 += 1
        elif cnt1 == 0:   c1, cnt1 = n, 1
        elif cnt2 == 0:   c2, cnt2 = n, 1
        else:             cnt1 -= 1; cnt2 -= 1
    threshold = len(nums) // 3
    return [c for c in (c1, c2) if c is not None and nums.count(c) > threshold]

# ── demo ──
print(sorted(majority_element_n3([3,2,3])))
print(sorted(majority_element_n3([1,1,1,3,3,2,2,2])))

## 6. Dutch National Flag (3-Way Partition)

**Concept:** Sort 0s, 1s, 2s in a single pass using three pointers: `lo` (boundary of 0s), `mid` (current), `hi` (boundary of 2s). Swap and advance accordingly.  

**Use when:** Sort array of 3 distinct values in one pass, partition around a pivot.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def sort_colors(nums: list[int]) -> list[int]:
    """
    Sort array of 0s, 1s, 2s in-place (single pass).
    Time O(n)  Space O(1)

    >>> sort_colors([2,0,2,1,1,0])
    [0, 0, 1, 1, 2, 2]
    """
    lo = mid = 0
    hi = len(nums) - 1
    while mid <= hi:
        if nums[mid] == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]
            lo += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[hi] = nums[hi], nums[mid]
            hi -= 1
    return nums

# ── demo ──
print(sort_colors([2,0,2,1,1,0]))

## 7. Merge Intervals

**Concept:** Sort intervals by start time, then greedily merge any overlapping pair by extending the end of the last merged interval.  

**Use when:** Calendar overlap, meeting rooms, range merging.  

**Time:** O(n log n) &nbsp;|&nbsp; **Space:** O(n)

In [ ]:
def merge_intervals(intervals: list[list[int]]) -> list[list[int]]:
    """
    Merge all overlapping intervals.
    Time O(n log n)  Space O(n)

    >>> merge_intervals([[1,3],[2,6],[8,10],[15,18]])
    [[1, 6], [8, 10], [15, 18]]
    """
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

# ── demo ──
print(merge_intervals([[1,3],[2,6],[8,10],[15,18]]))

In [ ]:
def insert_interval(
    intervals: list[list[int]], new: list[int]
) -> list[list[int]]:
    """
    Insert new interval into sorted non-overlapping intervals list.
    Time O(n)  Space O(n)

    >>> insert_interval([[1,3],[6,9]], [2,5])
    [[1, 5], [6, 9]]
    """
    result = []
    i = 0
    n = len(intervals)
    while i < n and intervals[i][1] < new[0]:
        result.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= new[1]:
        new[0] = min(new[0], intervals[i][0])
        new[1] = max(new[1], intervals[i][1])
        i += 1
    result.append(new)
    while i < n:
        result.append(intervals[i]); i += 1
    return result

# ── demo ──
print(insert_interval([[1,3],[6,9]], [2,5]))

## 8. Longest Substring Without Repeating Characters

**Concept:** Sliding window with a hash map tracking last-seen character positions. When a duplicate is found, jump the left pointer past the previous occurrence.  

**Use when:** Longest substring with unique chars, any "at most k distinct" variant.  

**Time:** O(n) &nbsp;|&nbsp; **Space:** O(min(n, alphabet))

In [ ]:
def length_of_longest_substring(s: str) -> int:
    """
    Time O(n)  Space O(min(n, alphabet))

    >>> length_of_longest_substring("abcabcbb")
    3
    >>> length_of_longest_substring("pwwkew")
    3
    """
    last_seen: dict[str, int] = {}
    lo = best = 0
    for hi, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= lo:
            lo = last_seen[ch] + 1
        last_seen[ch] = hi
        best = max(best, hi - lo + 1)
    return best

# ── demo ──
print(length_of_longest_substring("abcabcbb"))
print(length_of_longest_substring("pwwkew"))

## 9. Minimum Window Substring

**Concept:** Expand the right pointer to include all required characters, then shrink the left pointer to minimize the window. Track a `missing` counter to know when the window is valid.  

**Use when:** Smallest window containing all characters of a pattern.  

**Time:** O(|s| + |t|) &nbsp;|&nbsp; **Space:** O(|t|)

In [ ]:
def min_window_substring(s: str, t: str) -> str:
    """
    Smallest window in s containing all characters of t.
    Time O(|s| + |t|)  Space O(|t|)

    >>> min_window_substring("ADOBECODEBANC", "ABC")
    'BANC'
    >>> min_window_substring("a", "a")
    'a'
    """
    if not t or not s:
        return ""
    need = Counter(t)
    missing = len(t)
    lo = best_lo = 0
    best_len = float("inf")

    for hi, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1

        if missing == 0:
            while need[s[lo]] < 0:
                need[s[lo]] += 1
                lo += 1
            if hi - lo + 1 < best_len:
                best_len = hi - lo + 1
                best_lo = lo
            need[s[lo]] += 1
            missing += 1
            lo += 1

    return s[best_lo: best_lo + best_len] if best_len < float("inf") else ""

# ── demo ──
print(min_window_substring("ADOBECODEBANC", "ABC"))
print(min_window_substring("a", "a"))

## 10. Matrix — Spiral Order & Rotate 90°

**Concept:** Spiral: layer-by-layer traversal shrinking boundaries after each direction. Rotate 90°: transpose (swap `[i][j]` with `[j][i]`) then reverse each row — both in-place.  

**Use when:** Matrix traversal problems, image rotation.  

**Time:** O(m·n) &nbsp;|&nbsp; **Space:** O(1)

In [ ]:
def spiral_order(matrix: list[list[int]]) -> list[int]:
    """
    Traverse matrix in spiral (clockwise) order.
    Time O(m*n)  Space O(1) extra

    >>> spiral_order([[1,2,3],[4,5,6],[7,8,9]])
    [1, 2, 3, 6, 9, 8, 7, 4, 5]
    """
    result = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):       result.append(matrix[top][c])
        top += 1
        for r in range(top, bottom + 1):       result.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1): result.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1): result.append(matrix[r][left])
            left += 1
    return result

# ── demo ──
print(spiral_order([[1,2,3],[4,5,6],[7,8,9]]))

In [ ]:
def rotate_matrix_90(matrix: list[list[int]]) -> list[list[int]]:
    """
    Rotate n×n matrix 90° clockwise in-place.
    Step 1: transpose  Step 2: reverse each row
    Time O(n²)  Space O(1)

    >>> rotate_matrix_90([[1,2,3],[4,5,6],[7,8,9]])
    [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
    """
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()
    return matrix

# ── demo ──
print(rotate_matrix_90([[1,2,3],[4,5,6],[7,8,9]]))

In [ ]:
if __name__ == "__main__":
    import doctest
    results = doctest.testmod(verbose=False)
    print(f"Arrays & Strings: {results.attempted} tests, {results.failed} failed")

    assert max_subarray_with_indices([-2,1,-3,4,-1,2,1,-5,4]) == (6, 3, 6)
    assert majority_element_n3([1,1,1,3,3,2,2,2]) == [1, 2]
    print("All assertions passed.")